In [ ]:
import pandas as pd
import numpy as np

import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
import torch.autograd as autograd
import matplotlib.pyplot as plt
import time
import math
from torch.optim.lr_scheduler import StepLR 
from scipy.optimize import minimize
from scipy.stats import norm
import warnings   
 
import itertools

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors


from lifelines import CoxPHFitter

import warnings
from lifelines.utils import ConvergenceWarning, ConvergenceError

 


import pymoo
 
from pymoo.core.problem import ElementwiseProblem

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.termination import get_termination
 
from utils import *

In [ ]:

df = pd.read_csv("demo_data.csv")
print(df.shape)
    
# --- Fixed and relaxed criteria ---
#FC: fixed criteria
#RC: potential relaxed criteria
criteria_fixed = ['passes_FC1', 'passes_FC2']
criteria_relaxed =[  "passes_RC1", "passes_RC2", "passes_RC3", "passes_RC4",
  "passes_RC5", "passes_RC6", "passes_RC7",
  "passes_RC8", "passes_RC9", "passes_RC10"]
print(len(criteria_relaxed))


# --- Criteria columns ---
criteria_cols = criteria_fixed +criteria_relaxed  

for col in criteria_cols:
    pass_count = df[col].sum(skipna=True)
    print(f"{col}: {pass_count} / {len(df)} patients pass")

# --- Generate subgroups (placeholder for custom function) ---
subgroups = generate_subgroups(criteria_relaxed, criteria_fixed)
print(subgroups)

## Evaluation across eligibility configurations

In [ ]:
 
 
for i  in range(  len(subgroups)): 
    torch.manual_seed(26) 
    cri = 'C'+str(i) 
    print('===============', f"This is the C{i}-th configuration", '===============' )  
    criteria = subgroups.loc[i, "criteria"].split(", ")
    # Select rows where all criteria columns == 1
    mask = (df[criteria].sum(axis=1) == len(criteria))
    # Return filtered dataframes
    df_sub = df.loc[mask].copy()

      
    ### UNO pipeline to df_sub
    # --- Define covariates and outcomes ---
    column_names_list = list(df_sub.columns)
    cov_name = []

    xvars = [c for c in df_sub.columns if c.startswith("cov")]
    x_df  = pd.get_dummies(df_sub[xvars], drop_first=True) 
    threshold = 0.01  
    low_var_cols = [col for col in x_df.columns if (x_df[col].var() < threshold)   ] 
    # Drop them
    x_df  = x_df.drop(columns=low_var_cols)
    X = x_df.astype(np.float32).to_numpy()  
    Xt = torch.tensor(X, dtype=torch.float32)

    A =  df_sub['trt'].to_numpy()  
    At = torch.tensor(A.reshape(-1, 1), dtype=torch.float32)  
    print(df_sub.shape)


    ### PS prediction
    d = X.shape[1]
    n_hidden =  80  
    n_depth = 2
    model = PS(in_N=d , m= n_hidden, depth= n_depth)   
    model.apply(weights_init)
    print(model) 
    print(sum(p.numel() for p in model.parameters() if p.requires_grad))

    criterion = nn.BCELoss()  
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-6)
    scheduler = StepLR(optimizer, step_size=10, gamma=0.5)    
    n_epochs = 30
    batch_size = 32
    #
    model.eval()   
    with torch.no_grad():   
        test_outputs = model(Xt).squeeze()   
        predicted_prob = test_outputs.numpy()
    actual_treatment = At.numpy().squeeze()
    pred = (predicted_prob >=0.5)
    print('initialzation', np.sum(pred==actual_treatment )/pred.shape[0],optimizer.param_groups[-1]['lr'])
    #
    for epoch in range(n_epochs):  
        model.train()  
        running_loss = 0.0
        permutation = torch.randperm(Xt.size(0)) 
        for i in range(0, Xt.shape[0], batch_size):
            ind = permutation[i:i + batch_size]
            batch_x, batch_a = Xt[ind], At[ind]
            out = model(batch_x)  
            optimizer.zero_grad()
            loss = criterion(out,batch_a)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()

        model.eval()   
        with torch.no_grad():   
            test_outputs = model(Xt).squeeze()   
            predicted_prob = test_outputs.numpy()
        actual_treatment = At.numpy().squeeze()
        pred = (predicted_prob >=0.5) 
        if (epoch + 1) %10 == 0:
            print(f'Epoch {epoch+1}/{n_epochs}, Loss: {running_loss/len(Xt)*batch_size}', np.sum(pred==actual_treatment )/pred.shape[0],optimizer.param_groups[-1]['lr'])
        if np.sum(pred==actual_treatment )/pred.shape[0]>0.95:
            break
    # 
    ### NCO prediction
    NCO = []
    j=0 
    ncovars = [c for c in df_sub.columns if c.startswith("nco")]
    for i in ncovars: 
        j += 1
        W =  df_sub[i].to_numpy()  
        if np.mean(W[A==0])>0.001 and np.mean(W[A==1])>0.001:  
            NCO.append(i)   
    print(j, len(NCO ))

    #use Cox regression for each W on A:  using propensity score
    model.eval()   
    with torch.no_grad():
        ps = model(Xt).numpy().squeeze()   
     
    #NCO slection and random split
    logHR_L = []
    se_L = []
    index_all = []
    for i in NCO: 
        ##
        W =  df_sub[i] 
        W_time = df_sub['time_'+i] 
        data = pd.DataFrame({
            'event': W,
            'time': W_time,
            'treatment': A, 
        }) 
        data['ps_strata'] = pd.qcut(ps, q=5, labels=False, duplicates='drop')
        cph = CoxPHFitter( ) 
         
         
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            try:
                cph.fit(data, duration_col='time', event_col='event', strata=['ps_strata'], robust=True ) 
            # --- catch lifelines exceptions ---
            except ConvergenceError as e:
                print(f"Skipping {i} due to ConvergenceError: {e}")
                continue

            except Exception as e:
                print(f"Skipping {i} due to other exception: {e}")
                continue
            # Detect RuntimeWarnings or ConvergenceWarnings
            if any(
                issubclass(warn.category, RuntimeWarning) or 
                issubclass(warn.category, ConvergenceWarning)
                for warn in w):
                print(f"Skipping outcome {i} due to warning.")
                continue

            
        # log hazard ratio (coefficient)
        logHR = cph.params_["treatment"]
        # standard error of logHR
        se_logHR = cph.standard_errors_["treatment"]
        # 95% CI for logHR
        logHR_lower = logHR - 1.96 * se_logHR
        logHR_upper = logHR + 1.96 * se_logHR

        index_all.append(i) 
        logHR_L.append( logHR) 
        se_L.append(se_logHR)

     
    logHr = np.array(logHR_L )  
    seLogHr =np.array(se_L )   
    null_distribution = fit_null(logHr, seLogHr)
    EASE=compute_expected_absolute_systematic_error_null(null_distribution)
    W_all = [i for i in index_all] 
    print(len(W_all), W_all)
    index_split = random.sample(range(len(W_all)), math.ceil( len(W_all)/3 ))  
    W_split = [W_all[i] for i in index_split]
    print(len(W_split ), W_split )

    #NCO prediction
    nco_dict = {}
    NCO_select = W_split
    j = 0
    for i in NCO_select:  
        W =  df_sub[i].to_numpy() 
        #treatment 1
        lr1 = LogisticRegression(max_iter=500, solver='liblinear')
        lr1.fit(  X[A==1 ],W[A==1] ) 
        nco1 = lr1.predict_proba(X)[:,1]
            
        #treatment 0
        lr0 = LogisticRegression(max_iter=500, solver='liblinear')
        lr0.fit(  X[A==0 ],W[A==0] ) 
        nco0 = lr1.predict_proba(X)[:,1]
            
        #print('-----------------------'+str(j))
        NCO1_pred = test_NCO(lr1, X , W )
        NCO0_pred = test_NCO(lr0, X , W )
        #print('-----------------------')
        #compare
        nco_dict[str(i)+'nco1_p'] = NCO1_pred
        nco_dict[str(i)+'nco0_p'] = NCO0_pred
        j += 1
    # 
        
    #unweighted median
    for i in NCO_select:  
        W =  df_sub[i].to_numpy()  
        NCO_pred0  =   nco_dict[str(i)+'nco0_p'] 
        NCO_pred1  =  nco_dict[str(i)+'nco1_p'] 
        #ind = (abs(NCO_pred0-NCO_pred1)>0.01).reshape(-1,1) 
        diff = np.abs(NCO_pred0 - NCO_pred1) 
        threshold = np.quantile(diff, 0.9)
        ind = (diff >= threshold).astype(int).reshape(-1, 1)
        if i == NCO_select[0]: 
            I = ind
        else:
            I = np.hstack((I,ind))
    print(I.shape)
    avg_I = np.sum(I,axis=1)  
    print(np.max(avg_I))
    con = (avg_I>=(np.max(avg_I)//2 +1   ) ) # 
    X_h = Xt[con]
    X_l = Xt[~con]
    A_h = At[con]
    A_l = At[~con]
    print(X_h.shape, X_l.shape,A_h.shape, A_l.shape)
    print('high biased ratio: ', X_h.shape[0]/avg_I.shape[0])

    if X_h.shape[0]/avg_I.shape[0]>=0.45:
        print('high biased ratio is too high: ',  X_h.shape[0]/avg_I.shape[0])




    ### compute EASE: expected_absolute_systematic_error
    print(len(W_all))
    W_left = []
    for i in W_all:
        if i not in W_split:
            W_left.append(i) 
    ### 
    index_split = random.sample(range(len(W_left)), math.ceil( len(W_left)/4 ))  
    W_p = [W_left[i] for i in index_split] #for pruning ratio selection
    W_pm = [] #for ease calculation
    for i in W_left:
        if i not in W_p:
            W_pm.append(i) 
    print(len(W_all), len(W_left), len(W_p), len(W_pm))


    W_select_ease = W_p +W_split  #####W_left
    print(len(W_select_ease))

     
    ###########

    trained =  model.state_dict()   
    ratio_range = [ 0.4,0.45, 0.5,0.55, 0.6, 0.65,0.7 ]
    model_list = []
    EASE_p_list = [] 

    for ratio in ratio_range:
        ### Pruning on PS model
        model_p = M_pruned(in_N=d , m= n_hidden, depth= n_depth)   
        model_p.load_state_dict(trained)

        #Test
        model_p.eval()   
        with torch.no_grad():   
            test_outputs, p_list  = model_p.pre_act(Xt  )   
            test_outputs = test_outputs.squeeze()
            predicted_prob = test_outputs.numpy() 
            actual_treatment = At.numpy().squeeze()
        pred = (predicted_prob >=0.5)
        p_list_l = [item[~con] for item in p_list]
        p_list_h = [item[con] for item in p_list]

        #L2 for each neuron
        epsilon = 1e-3#1e-12
        sl = []
        for i in range(len(p_list_l)): 
            norm1_l =  torch.mean( p_list_l[i]**2 , dim=0)
            norm1_h =  torch.mean( p_list_h[i]**2 , dim=0) 
            score = torch.div(norm1_h, norm1_l+epsilon)
            sl.append(score)

        sm = torch.cat([i.unsqueeze(1) for i in sl], dim=1) #score matrix
        #print(torch.max(sm), torch.min(sm))
        num_neuron = n_hidden   
        num_to_prune = int(num_neuron* n_depth * ratio)
        values, indices = torch.topk(sm.flatten(), num_to_prune)
        row_indices = indices // sm.size(1)   
        col_indices = indices % sm.size(1)  

        state_p = model_p.state_dict()
        with torch.no_grad():
            state_p['stack.'+str(0)+'.weight'][row_indices[col_indices==0],:]=0
            state_p['stack.'+str(0)+'.bias'][row_indices[col_indices==0]]=0
            state_p['stack.'+str(1)+'.weight'][row_indices[col_indices==1],:]=0
            state_p['stack.'+str(1)+'.weight'][:,row_indices[col_indices==0]]=0
            state_p['stack.'+str(1)+'.bias'][row_indices[col_indices==1]]=0  
            state_p['stack.'+str(2)+'.weight'][:,row_indices[col_indices==2]]=0
            model_p.load_state_dict(state_p)

        
        ### finetuning on low biased data after pruning
        criterion = nn.BCELoss()  
        optimizer = optim.Adam(model_p.parameters(), lr=0.0005, weight_decay=5e-6 )
        scheduler = StepLR(optimizer, step_size=10, gamma=0.5)    

        n_epochs =  10 
        #
        model_p.eval()   
        with torch.no_grad():   
            test_outputs = model_p(X_l).squeeze()  
            predicted_prob = test_outputs.numpy()
            
        actual_treatment = A_l.numpy().squeeze()
        pred = (predicted_prob >=0.5) 
        #
        for epoch in range(n_epochs):  
            model_p.train()  
            running_loss = 0.0
            permutation = torch.randperm(X_l.size(0)) 
            for i in range(0, X_l.shape[0], batch_size):

                ind = permutation[i:i + batch_size]
                batch_x, batch_a = X_l[ind], A_l[ind]
                out = model_p(batch_x)  
                optimizer.zero_grad()
                loss = criterion(out,batch_a)
                loss.backward()
                # Manually prevent weight updates for frozen neurons
                with torch.no_grad():
                    model_p.stack[0].weight.grad[row_indices[col_indices==0],:]=0
                    model_p.stack[0].bias.grad[row_indices[col_indices==0]]=0
                    model_p.stack[1].weight.grad[row_indices[col_indices==1],:]=0
                    model_p.stack[1].weight.grad[:,row_indices[col_indices==0]]=0
                    model_p.stack[1].bias.grad[row_indices[col_indices==1]]=0 
                    model_p.stack[2].weight.grad[:,row_indices[col_indices==2]]=0
                #--------------------
                optimizer.step()
                running_loss += loss.item()
            scheduler.step()

            model_p.eval()   
            with torch.no_grad():   
                test_outputs = model_p(Xt).squeeze()  
                predicted_prob = test_outputs.numpy()
            actual_treatment = At.numpy().squeeze()
            pred = (predicted_prob >=0.5) 
                 
        

        #after pruning
        model_p.eval()   
        with torch.no_grad():
            ps = model_p(Xt).numpy().squeeze()
 
        logRR_L = []
        se_L = []
        for i in W_select_ease:  
            W =  df_sub[i] 
            W_time = df_sub['time_'+i ] 
            data = pd.DataFrame({
                'event': W,
                'time': W_time,
                'treatment': A, 
            })  
            data['ps_strata'] = pd.qcut(ps, q=5, labels=False, duplicates='drop' )
            cph = CoxPHFitter( ) 
         
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                try:
                    cph.fit(data, duration_col='time', event_col='event', strata=['ps_strata'],robust=True ) 
                # --- catch lifelines exceptions ---
                except ConvergenceError as e:
                    print(f"Skipping {i} due to ConvergenceError: {e}")
                    continue

                except Exception as e:
                    print(f"Skipping {i} due to other exception: {e}")
                    continue
                # Detect RuntimeWarnings or ConvergenceWarnings
                if any(
                    issubclass(warn.category, RuntimeWarning) or 
                    issubclass(warn.category, ConvergenceWarning)
                    for warn in w):
                    print(f"Skipping outcome {i} due to warning.")
                    continue
            # log hazard ratio (coefficient)
            logHR = cph.params_["treatment"]
            # standard error of logHR
            se_logHR = cph.standard_errors_["treatment"]
            logRR_L.append( logHR) 
            se_L.append(se_logHR)

        logRr_p = np.array(logRR_L) 
        seLogRr_p =np.array(se_L)   
        null_distribution = fit_null(logRr_p, seLogRr_p)
        EASE_p =compute_expected_absolute_systematic_error_null(null_distribution) 
   
        EASE_p_list.append(EASE_p) 
        model_list.append(model_p.state_dict()) 
            

    print(EASE_p_list)
    best = np.argsort( np.array(EASE_p_list))[0]
    ratio = ratio_range[best]
    print('best:', ratio, EASE_p_list[best])

    model_p.load_state_dict(model_list[best])

    ####### Cox HR 
    model_p.eval()   
    with torch.no_grad():
        ps = model_p(Xt).numpy().squeeze() 


    W =  df_sub['event_DateOfDeath']
    W_time = df_sub['time_DateOfDeath']
    data = pd.DataFrame({
        'event': W,
        'time': W_time,
        'treatment': A, 
    })  
    data['ps_strata'] = pd.qcut(ps, q=5, labels=False, duplicates='drop')
    cph = CoxPHFitter( ) 
     
 
    with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            try:
                cph.fit(data, duration_col='time', event_col='event', strata=['ps_strata'], robust=True ) 
            # --- catch lifelines exceptions ---
            except ConvergenceError as e:
                print(f"Skipping {i} due to ConvergenceError: {e}")
                continue

            except Exception as e:
                print(f"Skipping {i} due to other exception: {e}")
                continue
            # Detect RuntimeWarnings or ConvergenceWarnings
            if any(
                issubclass(warn.category, RuntimeWarning) or 
                issubclass(warn.category, ConvergenceWarning)
                for warn in w):
                print(f"Skipping outcome {i} due to warning.")
                continue
    # log hazard ratio (coefficient)
    logHR_p = cph.params_["treatment"]
    # standard error of logHR
    se_logHR_p = cph.standard_errors_["treatment"]
    print('UNO logHR and se: ', logHR_p,  se_logHR_p )
    print('UNO HR: ',np.exp(logHR_p))

 
 
     
    W_select_ease = W_pm  
 
    model_p.load_state_dict(model_list[best])
    model_p.eval()   
    with torch.no_grad():
        ps_p = model_p(Xt).numpy().squeeze() 

    logRR_L = []
    se_L = []
    for i in W_select_ease:  
        W =  df_sub[i] 
        W_time = df_sub['time_'+i ] 
        data = pd.DataFrame({
            'event': W,
            'time': W_time,
            'treatment': A, 
        })  
        data['ps_strata'] = pd.qcut(ps_p, q=5, labels=False, duplicates='drop')
        cph = CoxPHFitter( ) 
         
         
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            try:
                cph.fit(data, duration_col='time', event_col='event', strata=['ps_strata'], robust=True) 
            # --- catch lifelines exceptions ---
            except ConvergenceError as e:
                print(f"Skipping {i} due to ConvergenceError: {e}")
                continue

            except Exception as e:
                print(f"Skipping {i} due to other exception: {e}")
                continue
            # Detect RuntimeWarnings or ConvergenceWarnings
            if any(
                issubclass(warn.category, RuntimeWarning) or 
                issubclass(warn.category, ConvergenceWarning)
                for warn in w):
                print(f"Skipping outcome {i} due to warning.")
                continue
        # log hazard ratio (coefficient)
        logHR = cph.params_["treatment"]
        # standard error of logHR
        se_logHR = cph.standard_errors_["treatment"]
        logRR_L.append( logHR) 
        se_L.append(se_logHR)

    logRr = np.array(logRR_L)  
    seLogRr =np.array(se_L)   
    null_distribution = fit_null(logRr, seLogRr)
    EASE_23 =compute_expected_absolute_systematic_error_null(null_distribution) 
     

    #AE
    AE_new_onset_PP = AE_rates(df_sub)
    #G-index
    if cri == 'C0':
        gindex =1
        ps0 = ps_p
    else:
        gindex = calculate_g_index(ps0, ps_p, n_grid=512)
    print('G-index', gindex)

 


    #a single summary row
    if cri == 'C0':
        result_df = pd.DataFrame([{
            "subpop": cri,
            "logHR_uno": logHR_p,
            "seLogHR_uno": se_logHR_p,
            "n": df_sub.shape[0], 
            "AE_new_onset_PP": AE_new_onset_PP, 
            "EASE_uno": EASE_23,
            "gindex": gindex, 
        }])
    else:
        new_row = pd.DataFrame([{
            "subpop": cri, 
            "logHR_uno": logHR_p,
            "seLogHR_uno": se_logHR_p,
            "n": df_sub.shape[0], 
            "AE_new_onset_PP": AE_new_onset_PP,  
            "EASE_uno": EASE_23,
            "gindex": gindex, 
        }])

         
        result_df = pd.concat([result_df,  new_row ], ignore_index=True)
    #result_df.to_csv('results.csv')
    

## Multi-objective optimization

In [ ]:
result_df['HR_uno'] = np.exp(result_df['logHR_uno'])

if 'binary' not in result_df.columns:
    result_df['binary'] = [None] * len(result_df)
result_df[ 'binary'] = subgroups['binary_inclusion'].values
count_fixed = np.max(result_df["n"])
result_df["n_reduced"] = ( count_fixed-result_df["n"]  )/count_fixed

In [ ]:
 
lookup = {
    tuple(row['binary']): (row['HR_uno'],  row['n_reduced'] ) 
    for _, row in result_df.iterrows()
}


class BinaryProblem(ElementwiseProblem):
    def __init__(self, lookup): 
        self.lookup = lookup 
        self.n_var = len(result_df['binary'].iloc[0])
    
        super().__init__(
            n_var=self.n_var,
            n_obj=2, 
            xl=np.zeros(self.n_var),
            xu=np.ones(self.n_var)
        )

        # storage for logging
        self.history = {(1,1,1,1,1,1,1,1,1,1,1):(result_df.iloc[1]['HR_uno' ], result_df.iloc[1]['n_reduced' ])}       

    def _evaluate(self, x, out, *args, **kwargs):
        x_tuple = tuple(int(v) for v in x)
        f1, f2 = self.lookup.get(x_tuple, (9999, 9999))
        out["F"] = [f1, f2]

        # log the evaluation
        if x_tuple not in self.history:
            self.history[x_tuple] = (f1, f2)  


problem =  BinaryProblem(lookup)


In [ ]:
from pymoo.optimize import minimize

In [ ]:
algorithm = NSGA2(
    pop_size=100,  
    sampling=BinaryRandomSampling(),
    crossover=TwoPointCrossover(),
    mutation=BitflipMutation(prob=0.5)
)
res = minimize(problem, algorithm,('n_gen', 30),
               seed=1,
               save_history=True,
               verbose=True)


X = res.X
F = res.F